# Data Cleaning

## Executive Summary
- Load raw data
- Flatten nested JSON cols
- Drop cols with more than 80% data missing

Load raw data, check schema/types, handle nulls/duplicates
Basic EDA: distributions, date ranges, sanity checks
Export a cleaned dataset to data/processed/ for downstream notebooks

## Initial Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Project root = one level above the notebooks folder
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

sys.path.append(str(PROJECT_ROOT))
from src.preprocessing import *

## Load Raw Data

In [2]:
check_df = pd.read_csv(DATA_DIR / "train_v2.csv", nrows=20_000)
print("Loaded train_v2.csv. Shape:", check_df.shape)
check_df.head(1)

Loaded train_v2.csv. Shape: (20000, 13)


,channelGrouping,customDimensions,date,device,fullVisitorId,geoNetwork,hits,socialEngagementType,totals,trafficSource,visitId,visitNumber,visitStartTime
0,Organic Search,"[{'index': '4', 'value': 'EMEA'}]",20171016,"{""browser"": ""Firefox"", ""browserVersion"": ""not ...",3162355547410993243,"{""continent"": ""Europe"", ""subContinent"": ""Weste...","[{'hitNumber': '1', 'time': '0', 'hour': '17',...",Not Socially Engaged,"{""visits"": ""1"", ""hits"": ""1"", ""pageviews"": ""1"",...","{""campaign"": ""(not set)"", ""source"": ""google"", ...",1508198450,1,1508198450


**Decision**: The cell above shows what the first row looks like. Four features in the train & test datasets are in the JSON form - `device`, `geoNetwork`, `totals`, `trafficSource`. These features need to be flattened out. 

In [3]:
# create sample df with 20,000 rows for exploration purposes; original dataset's too large
sample_train_df = load_df(DATA_DIR / "train_v2.csv", nrows=20_000) 

Loaded train_v2.csv. Shape: (20000, 59)


**Initial approach (2024)**: In the initial analysis, the `customDimensions` and `hits` columns were excluded completely because they contained deeply nested hit-level data and were not yet evaluated for their potential analytical value.

**Revised approach (2026)**: After revisiting `customDimensions` and `hits`, I found potentially useful regional and behavioral information. I will extract relevant values from both during feature engineering.

In [4]:
sample_train_df[['customDimensions']].value_counts()

customDimensions                            
[{'index': '4', 'value': 'North America'}]      9288
[{'index': '4', 'value': 'EMEA'}]               3775
[]                                              3605
[{'index': '4', 'value': 'APAC'}]               2566
[{'index': '4', 'value': 'South America'}]       468
[{'index': '4', 'value': 'Central America'}]     298
Name: count, dtype: int64

In [5]:
sample_train_df[['hits']].value_counts()

hits                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [7]:
sample_train_df.head(5)

,channelGrouping,customDimensions,date,fullVisitorId,hits,socialEngagementType,visitId,visitNumber,visitStartTime,device.browser,...,trafficSource.keyword,trafficSource.adwordsClickInfo.criteriaParameters,trafficSource.referralPath,trafficSource.isTrueDirect,trafficSource.adContent,trafficSource.adwordsClickInfo.page,trafficSource.adwordsClickInfo.slot,trafficSource.adwordsClickInfo.gclId,trafficSource.adwordsClickInfo.adNetworkType,trafficSource.adwordsClickInfo.isVideoAd
0,Organic Search,"[{'index': '4', 'value': 'EMEA'}]",20171016,3162355547410993243,"[{'hitNumber': '1', 'time': '0', 'hour': '17',...",Not Socially Engaged,1508198450,1,1508198450,Firefox,...,water bottle,not available in demo dataset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Referral,"[{'index': '4', 'value': 'North America'}]",20171016,8934116514970143966,"[{'hitNumber': '1', 'time': '0', 'hour': '10',...",Not Socially Engaged,1508176307,6,1508176307,Chrome,...,NaN,not available in demo dataset,/a/google.com/transportation/mtv-services/bike...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Direct,"[{'index': '4', 'value': 'North America'}]",20171016,7992466427990357681,"[{'hitNumber': '1', 'time': '0', 'hour': '17',...",Not Socially Engaged,1508201613,1,1508201613,Chrome,...,NaN,not available in demo dataset,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN
3,Organic Search,"[{'index': '4', 'value': 'EMEA'}]",20171016,9075655783635761930,"[{'hitNumber': '1', 'time': '0', 'hour': '9', ...",Not Socially Engaged,1508169851,1,1508169851,Chrome,...,(not provided),not available in demo dataset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Organic Search,"[{'index': '4', 'value': 'Central America'}]",20171016,6960673291025684308,"[{'hitNumber': '1', 'time': '0', 'hour': '14',...",Not Socially Engaged,1508190552,1,1508190552,Chrome,...,(not provided),not available in demo dataset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Check for Missing Values

In [6]:
missing_report = get_missing_value_report(sample_train_df)
missing_report.head(20)

,column_name,percent_missing
0,totals.totalTransactionRevenue,99.035
1,totals.transactionRevenue,99.035
2,totals.transactions,99.020
3,trafficSource.adContent,97.395
4,trafficSource.adwordsClickInfo.adNetworkType,96.850
5,trafficSource.adwordsClickInfo.page,96.850
6,trafficSource.adwordsClickInfo.isVideoAd,96.850
7,trafficSource.adwordsClickInfo.slot,96.850
8,trafficSource.adwordsClickInfo.gclId,96.835
9,trafficSource.referralPath,69.480


In [23]:
sample_train_df[['trafficSource.isTrueDirect']].value_counts()

trafficSource.isTrueDirect
True                          6333
Name: count, dtype: int64